In [ ]:
# @title Instalare dependințe necesare
!pip install playwright pandas beautifulsoup4 nest_asyncio --quiet
!playwright install --with-deps chromium
print("✅ Instalarea s-a terminat cu succes! Poți rula următoarea celulă.")

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [ ]:
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

# Permitem rularea asincronă în mediul Google Colab
nest_asyncio.apply()

async def extrage_stiri_flashscore(url_principal, fisier_iesire="stiri_flashscore_curate.csv"):
    date_extrase = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )

        # Blocăm încărcarea imaginilor pentru viteză maximă
        await context.route("**/*", lambda route: route.abort()
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]
            else route.continue_())

        pagina_principala = await context.new_page()

        print(f"1. Accesăm pagina principală: {url_principal}")
        await pagina_principala.goto(url_principal, wait_until="domcontentloaded", timeout=60000)

        selector_articol = '[data-testid="wcl-newsArticlePreview"]'

        try:
            await pagina_principala.wait_for_selector(selector_articol, timeout=20000)
        except Exception:
            print("❌ Eroare: Nu s-au putut încărca elementele pe pagina principală.")
            await browser.close()
            return

        # --- PARTEA CORECTATĂ PENTRU EXTRAGEREA LINK-URILOR ---
        elemente_stiri = await pagina_principala.locator(selector_articol).all()
        linkuri_stiri = []

        for element in elemente_stiri:
            # 1. Încercăm să luăm href direct de pe elementul principal
            href = await element.get_attribute('href')

            # 2. Dacă elementul nu are href, abia atunci căutăm un tag <a> în interior
            if not href:
                try:
                    href = await element.locator('a').first.get_attribute('href', timeout=2000)
                except:
                    href = None

            if href:
                link_complet = href if href.startswith('http') else f"https://www.flashscore.ro{href}"
                if link_complet not in linkuri_stiri:
                    linkuri_stiri.append(link_complet)

        print(f"-> Am găsit {len(linkuri_stiri)} articole. Începem extragerea textului...\n")

        for link in linkuri_stiri:
            try:
                pagina_articol = await context.new_page()
                await pagina_articol.goto(link, wait_until="domcontentloaded", timeout=45000)

                await pagina_articol.wait_for_selector('p', timeout=15000)

                html_continut = await pagina_articol.content()
                soup = BeautifulSoup(html_continut, 'html.parser')

                paragrafe = soup.find_all('p')
                text_articol_complet = []

                for p in paragrafe:
                    # Funcția magică ce ignoră <a>...</a> dar păstrează cuvântul
                    text_curat = p.get_text(separator=" ", strip=True)
                    if text_curat:
                        text_articol_complet.append(text_curat)

                text_final = " \n".join(text_articol_complet)

                date_extrase.append({
                    "URL": link,
                    "Text_Stire": text_final
                })

                print(f"✅ Extras cu succes: {link}")
                await pagina_articol.close()

            except Exception as e:
                print(f"⚠️ Eroare ignorată la articolul {link}: {e}")

        await browser.close()

    if date_extrase:
        df = pd.DataFrame(date_extrase)
        df.to_csv(fisier_iesire, index=False, encoding='utf-8-sig')
        print(f"\n🎉 Finalizat! Am salvat datele în '{fisier_iesire}'")
        display(df.head())
    else:
        print("\n❌ Nu s-au putut extrage date.")

# Rulăm scriptul
URL_TINTA = 'https://www.flashscore.ro/stiri/cupa-mondiala/lvUBR5F8CdnS0XT8/page-10/'
asyncio.run(extrage_stiri_flashscore(URL_TINTA))

1. Accesăm pagina principală: https://www.flashscore.ro/stiri/cupa-mondiala/lvUBR5F8CdnS0XT8/page-10/
-> Am găsit 215 articole. Începem extragerea textului...

✅ Extras cu succes: https://www.flashscore.ro/stiri/fotbal-cupa-mondiala-2026-mexic-africa-de-sud-grupa-a-deschidere-rezultat-video/biMQUJy8/
✅ Extras cu succes: https://www.flashscore.ro/stiri/fotbal-cupa-mondiala-2026-live-ziua-1/fXOcB5s2/
✅ Extras cu succes: https://www.flashscore.ro/stiri/tenis-emma-raducanu-sorana-cirstea-ora-start-queens-jaqueline-cristian/Kv42f3L7/
✅ Extras cu succes: https://www.flashscore.ro/stiri/tenis-s-hertogenbosch-atp-simplu-tennis-tracker-cirstea-o-intalneste-pe-emma-raducanu-ruse-joaca-contra-mertens-jaquelin-vs-boulter/8GQkDRCk/
✅ Extras cu succes: https://www.flashscore.ro/stiri/kylian-mbappe-precaut-in-fata-debutului-cu-senegal-de-la-cupa-mondiala/CW5fg11c/
✅ Extras cu succes: https://www.flashscore.ro/stiri/fotbal-cupa-mondiala-toate-echipele-oficiale-de-26-de-jucatori-pentru-cupa-mondiala-fi

,URL,Text_Stire
0,https://www.flashscore.ro/stiri/fotbal-cupa-mo...,Legendarul stadion Azteca din Ciudad de Mexico...
1,https://www.flashscore.ro/stiri/fotbal-cupa-mo...,"Cupa Mondială a început joi, 11 iunie! Turneul..."
2,https://www.flashscore.ro/stiri/tenis-emma-rad...,Sorana Cîrstea și Jaqueline Cristian vor evolu...
3,https://www.flashscore.ro/stiri/tenis-s-hertog...,Cele mai importante rezultate din tenis pot fi...
4,https://www.flashscore.ro/stiri/kylian-mbappe-...,"Atacantul echipei naționale a Franței, Kylian ..."


In [1]:
import unicodedata
import nltk

# Descărcăm modulul care știe să despartă textul în propoziții
nltk.download('punkt', quiet=True)

# 1. DICȚIONARUL TĂU PRINCIPAL
dictionar_wc = {
    # --- FAVORITELE ȘI GIGANȚII ---
    "franta": ["mbappe", "cherki", "doue", "dembele", "tchouameni", "olise", "kante", "loris", "upamecano", "varane", "kounde", "konate", "saliba"],
    "argentina": ["messi", "alvarez", "paz", "emiliano martinez", "mac allister", "enzo fernandez", "de paul", "otamendi", "lautaro martinez", "simeone"],
    "brazilia": ["neymar", "vinicius", "endrick", "cunha", "casemiro", "alisson", "marquinhos", "paqueta", "rayan", "raphinha"],
    "portugalia": ["ronaldo", "bruno fernandes", "bernardo silva", "joao felix", "leao", "joao neves", "vitinha", "cancelo", "ruben dias", "dalot", "goncalo ramos"],
    "anglia": ["kane", "bellingham", "saka", "rogers", "rashford", "o'reilly", "rice", "pickford", "guehi", "gordon", "eze"],
    "spania": ["pedri", "gavi", "williams", "rodri", "yamal", "ferran torres", "eric garcia", "cucurella", "cubarsi", "raya"],
    "germania": ["musiala", "tah", "woltemade", "sane", "undav", "wirtz", "rudiger", "neuer", "havertz", "kimmich", "schlotterbeck", "stiller"],

    # --- ECHIPE SOLIDE / SURPRIZE ---
    "olanda": ["gakpo", "depay", "van dijk", "frenkie de jong", "dumfries", "ake", "de ligt", "van de ven"],
    "croatia": ["modric", "perisic", "kovacic", "gvardiol", "kramaric", "sucic"],
    "belgia": ["de bruyne", "lukaku", "doku", "courtois", "trossard", "tielemans", "lammens"],
    "uruguay": ["maximiliano araujo", "ugarte", "nunez", "gimenez", "ronald araujo", "bentancur"],
    "maroc": ["brahim diaz", "hakimi", "amrabat", "bounou", "rahimi"],
    "senegal": ["mane", "jackson", "diaraa", "camara", "mendy"],

    # --- ECHIPE CU 1-3 VEDETE MAJORE ---
    "columbia": ["luis diaz", "james rodriguez", "arias"],
    "norvegia": ["haaland", "odegaard", "sorloth"],
    "suedia": ["gyokeres", "isak", "kulusevski"],
    "turcia": ["calhanoglu", "arda guler", "yildiz"],
    "egipt": ["salah", "marmoush", "trezeguet"],
    "sua": ["pulisic", "mckennie", "balogun"],
    "elvetia": ["xhaka", "akanji", "sommer"],
    "coreea de sud": ["son", "hwang", "kim min-jae"],
    "mexic": ["alvarez", "gimenez", "ochoa"],
    "japonia": ["mitoma", "kubo", "endo"],
    "canada": ["davies", "david", "buchanan"],
    "ghana": ["kudus", "inaki williams", "partey"],
    "ecuador": ["caicedo", "hincapie", "valencia"],
    "coasta de fildes": ["kessie", "haller", "fofana"],
    "algeria": ["mahrez", "bennacer", "aouar"],
    "austria": ["sabitzer", "laimer", "arnautovic"],
    "cehia": ["soucek", "schick", "coufal"],
    "scotia": ["robertson", "mctominay", "mcginn"],
    "iran": ["taremi", "azmoun"],

    # --- ECHIPE ANALIZATE DOAR CA NAȚIUNE ---
    "arabia saudita": ["al dawsari"],
    "australia": ["ryan", "souttar"],
    "tunisia": ["msakni"],
    "paraguay": ["almiron", "enciso"],
    "qatar": ["afif", "ali"],
    "bosnia si hertegovina": ["dzeko", "demirovic"],
    "iordania": ["al-taamari"],
    "rd congo": ["mbemba", "wissa"],
    "uzbekistan": ["shomurodov"],
    "noua zeelanda": ["wood"],
    "africa de sud": [],
    "haiti": [],
    "curacao": [],
    "capul verde": [],
    "irak": [],
    "panama": []
}

# 2. HARTA INVERSĂ (Jucător -> Țară)
mapare_jucator_tara = {}
for tara, jucatori in dictionar_wc.items():
    for jucator in jucatori:
        mapare_jucator_tara[jucator] = tara

# 3. FUNCȚIA DE CURĂȚARE A TEXTULUI (Fără accente/diacritice)
def curata_text(text):
    text_normalizat = unicodedata.normalize('NFD', str(text))
    text_fara_diacritice = ''.join(c for c in text_normalizat if unicodedata.category(c) != 'Mn')
    return text_fara_diacritice.lower()

# 4. FUNCȚIA DE ÎMPĂRȚIRE A ȘTIRII ÎN PROPOZIȚII
def imparte_in_propozitii(text):
    return nltk.tokenize.sent_tokenize(text)

print("✅ Dicționarul, harta inversă și funcțiile de procesare au fost încărcate cu succes!")

✅ Dicționarul, harta inversă și funcțiile de procesare au fost încărcate cu succes!


In [2]:
# @title Celula Unificată: Analiză de Sentiment și Export Ierarhic (RATING_FINAL)
!pip install transformers torch tqdm openpyxl --quiet

import pandas as pd
import re
import nltk
from transformers import pipeline
from tqdm.notebook import tqdm

# Asigurăm pachetele de punctuație
nltk.download('punkt_tab', quiet=True)

print("⏳ Încărcăm modelul AI pe placa video (GPU)...")
analizor_sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    max_length=512,
    truncation=True,
    device=0  # Forțăm utilizarea GPU-ului pentru viteză maximă
)
print("✅ Modelul a fost încărcat cu succes!\n")

# 1. Încărcăm știrile extrase anterior
fisier_intrare = "/stiri_flashscore_curate.csv"

try:
    df_stiri = pd.read_csv(fisier_intrare)
    print(f"Am încărcat {len(df_stiri)} știri pentru analiză.")
except FileNotFoundError:
    print(f"❌ Eroare: Nu găsesc fișierul '{fisier_intrare}'.")
    df_stiri = pd.DataFrame()

rezultate_finale = []

if not df_stiri.empty:
    print("🔍 Începem citirea și notarea fiecărei propoziții...\n")

    for index, rand in tqdm(df_stiri.iterrows(), total=len(df_stiri)):
        url = rand['URL']
        text_complet = str(rand['Text_Stire'])
        scoruri_articol = {}

        propozitii = imparte_in_propozitii(text_complet)

        for prop in propozitii:
            prop_curata = curata_text(prop)
            jucatori_mentionati = set()
            tari_mentionate = set()

            for jucator, tara in mapare_jucator_tara.items():
                if re.search(r'\b' + re.escape(jucator) + r'\b', prop_curata):
                    jucatori_mentionati.add(jucator)
                    tari_mentionate.add(tara)

            for tara in dictionar_wc.keys():
                if re.search(r'\b' + re.escape(tara) + r'\b', prop_curata):
                    tari_mentionate.add(tara)

            if jucatori_mentionati or tari_mentionate:
                rezultat_ai = analizor_sentiment(prop)[0]
                eticheta = rezultat_ai['label']
                scor_incredere = rezultat_ai['score']

                if eticheta == 'positive':
                    valoare = scor_incredere
                elif eticheta == 'negative':
                    valoare = -scor_incredere
                else:
                    valoare = 0.0

                for j in jucatori_mentionati:
                    scoruri_articol[j] = scoruri_articol.get(j, 0.0) + valoare
                for t in tari_mentionate:
                    scoruri_articol[t] = scoruri_articol.get(t, 0.0) + valoare

        if scoruri_articol:
            rand_rezultat = {'URL': url, 'Text_Stire': text_complet}
            rand_rezultat.update(scoruri_articol)
            rezultate_finale.append(rand_rezultat)

    # 2. Asamblarea și Ordonarea Ierarhică a Tabelului Final
    if rezultate_finale:
        # Creăm dataframe-ul brut
        df_final = pd.DataFrame(rezultate_finale)

        # --- LOGICA DE ORDONARE ---
        coloane_fixe = ['URL', 'Text_Stire']
        coloane_ordonate = []

        mapare_structurata = {}
        for jucator, tara in mapare_jucator_tara.items():
            if tara not in mapare_structurata:
                mapare_structurata[tara] = []
            mapare_structurata[tara].append(jucator)

        for tara, jucatori in mapare_structurata.items():
            coloane_ordonate.append(tara)
            coloane_ordonate.extend(jucatori)

        ordine_finala = coloane_fixe + coloane_ordonate

        # Reordonăm forțat coloanele și completăm cu 0 unde nu există date
        df_final_ordonat = df_final.reindex(columns=ordine_finala).fillna(0.0)

        # --- CURĂȚARE TEXT PENTRU EXCEL ---
        # Înlocuim Enter-urile cu un spațiu simplu ca să nu se spargă rândurile
        df_final_ordonat['Text_Stire'] = df_final_ordonat['Text_Stire'].replace(r'\n|\r', ' ', regex=True)

        # --- EXPORT FINAL ---
        nume_fisier_final = "RATING_FINAL.xlsx"
        df_final_ordonat.to_excel(nume_fisier_final, index=False)

        print(f"\n🎉 GATA! Tabelul a fost salvat perfect ordonat sub numele: {nume_fisier_final}")
        print("Structura coloanelor a fost aranjată ierarhic (Țară -> Jucători).")

        # Afișăm un preview scurt
        display(df_final_ordonat.head())
    else:
        print("\n⚠️ Nu a fost detectat niciun jucător sau țară din dicționar.")

⏳ Încărcăm modelul AI pe placa video (GPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

✅ Modelul a fost încărcat cu succes!

Am încărcat 215 știri pentru analiză.
🔍 Începem citirea și notarea fiecărei propoziții...



  0%|          | 0/215 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



🎉 GATA! Tabelul a fost salvat perfect ordonat sub numele: RATING_FINAL.xlsx
Structura coloanelor a fost aranjată ierarhic (Țară -> Jucători).


,URL,Text_Stire,franta,mbappe,cherki,doue,dembele,tchouameni,olise,kante,...,demirovic,iordania,al-taamari,rd congo,mbemba,wissa,uzbekistan,shomurodov,noua zeelanda,wood
0,https://www.flashscore.ro/stiri/fotbal-cupa-mo...,Legendarul stadion Azteca din Ciudad de Mexico...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,https://www.flashscore.ro/stiri/fotbal-cupa-mo...,"Cupa Mondială a început joi, 11 iunie! Turneul...",0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,https://www.flashscore.ro/stiri/tenis-emma-rad...,Sorana Cîrstea și Jaqueline Cristian vor evolu...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,https://www.flashscore.ro/stiri/kylian-mbappe-...,"Atacantul echipei naționale a Franței, Kylian ...",3.401077,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.575127,0.0,0.0,0.0,0.0,0.0,0.0
4,https://www.flashscore.ro/stiri/fotbal-cupa-mo...,Cupa Mondială FIFA este la mai puțin de două s...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# @title Celula: Grafic Interactiv (Fără Suprapunere)
!pip install plotly pandas numpy --quiet

import pandas as pd
import numpy as np
import plotly.express as px

# 1. Lista țărilor
lista_tari = [
    "franta", "argentina", "brazilia", "portugalia", "anglia", "spania", "germania",
    "olanda", "croatia", "belgia", "uruguay", "maroc", "senegal", "columbia",
    "norvegia", "suedia", "turcia", "egipt", "sua", "elvetia", "coreea de sud",
    "mexic", "japonia", "canada", "ghana", "ecuador", "coasta de fildes", "algeria",
    "austria", "cehia", "scotia", "iran", "arabia saudita", "australia", "tunisia",
    "paraguay", "qatar", "bosnia si hertegovina", "iordania", "rd congo", "uzbekistan",
    "noua zeelanda", "africa de sud", "haiti", "curacao", "capul verde", "irak", "panama"
]

# 2. Încărcăm datele
try:
    df = pd.read_excel("RATING_FINAL.xlsx")
except FileNotFoundError:
    print("❌ Eroare: Nu găsesc fișierul RATING_FINAL.xlsx.")

# 3. Calculăm scorurile
scoruri_tari = []
for tara in lista_tari:
    if tara in df.columns:
        scor_total = df[tara].sum()
        if scor_total > 0:
            scoruri_tari.append({'Tara': tara.capitalize(), 'Scor_Rating': scor_total})

df_viz = pd.DataFrame(scoruri_tari).sort_values('Scor_Rating', ascending=False).reset_index(drop=True)

# 4. Algoritm pentru evitarea suprapunerii (Pack Circles)
# Calculăm o rază vizuală bazată pe rădăcina pătrată a scorului
def pack_circles(data):
    # Sortăm descrescător pentru a pune cele mai mari în centru
    radii = np.sqrt(data['Scor_Rating'].values)
    x = np.zeros(len(radii))
    y = np.zeros(len(radii))

    # Prima bulă e la (0,0)
    for i in range(1, len(radii)):
        # Căutăm un unghi și o distanță minimă care să nu se suprapună cu cele plasate deja
        angle = i * 137.508  # Golden angle
        # Începem de la o rază mică și creștem până nu mai avem coliziune
        r_pos = radii[0] * 0.5
        collision = True
        while collision:
            r_pos += 0.2
            curr_x = r_pos * np.cos(np.radians(angle))
            curr_y = r_pos * np.sin(np.radians(angle))

            collision = False
            for j in range(i):
                dist = np.sqrt((curr_x - x[j])**2 + (curr_y - y[j])**2)
                if dist < (radii[i] + radii[j]) * 1.05: # Factor de siguranță de 5%
                    collision = True
                    break
            x[i], y[i] = curr_x, curr_y
    return x, y

df_viz['X'], df_viz['Y'] = pack_circles(df_viz)
df_viz['Scor_Afisat'] = df_viz['Scor_Rating'].round(2)

# 5. Generarea graficului
fig = px.scatter(
    df_viz, x="X", y="Y",
    size="Scor_Rating",
    color="Tara",
    text="Tara",
    hover_name="Tara",
    hover_data={"Scor_Afisat": True, "X": False, "Y": False, "Scor_Rating": False, "Tara": False},
    size_max=70, # Ajustat pentru a preveni coliziunea vizuală
    template="plotly_dark",
    title="🌍 Ierarhia Echipelor Naționale 🌍"
)

fig.update_xaxes(visible=False, range=[df_viz['X'].min()-10, df_viz['X'].max()+10])
fig.update_yaxes(visible=False, range=[df_viz['Y'].min()-10, df_viz['Y'].max()+10])
fig.update_traces(textposition='middle center', textfont_size=10, textfont_color='white')
fig.update_layout(
    showlegend=False,
    margin=dict(t=50, b=0, l=0, r=0),
    hoverlabel=dict(bgcolor="white", font_size=14)
)

fig.show()
###########################################################
# @title Celula: Clasament Descrescător Favorite
import plotly.express as px

# Pregătim datele pentru a avea doar coloana finală dorită
df_bar_viz = df_viz.copy()
df_bar_viz['Scor'] = df_bar_viz['Scor_Rating'].round(2)

# Generăm graficul folosind noua coloană 'Scor'
fig_bar = px.bar(
    df_bar_viz,
    x="Tara",
    y="Scor",
    color="Scor",
    text="Scor",
    color_continuous_scale="Viridis",
    template="plotly_dark",
    title="📊 Clasamentul Favoritelor 📊",
    labels={"Scor": "Scor", "Tara": "Echipa"}
)

fig_bar.update_traces(texttemplate='%{text}', textposition='outside')
fig_bar.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
    coloraxis_showscale=False,
    margin=dict(t=50, b=100, l=50, r=50)
)

fig_bar.show()

In [22]:
# @title Celula: Grafic Interactiv Top 25 Jucători (Fără Suprapunere)
!pip install plotly pandas numpy --quiet

import pandas as pd
import numpy as np
import plotly.express as px

# 1. Lista țărilor (o folosim ca "blacklist" pentru a ști să o ignorăm și să extragem doar jucătorii)
lista_tari = [
    "franta", "argentina", "brazilia", "portugalia", "anglia", "spania", "germania",
    "olanda", "croatia", "belgia", "uruguay", "maroc", "senegal", "columbia",
    "norvegia", "suedia", "turcia", "egipt", "sua", "elvetia", "coreea de sud",
    "mexic", "japonia", "canada", "ghana", "ecuador", "coasta de fildes", "algeria",
    "austria", "cehia", "scotia", "iran", "arabia saudita", "australia", "tunisia",
    "paraguay", "qatar", "bosnia si hertegovina", "iordania", "rd congo", "uzbekistan",
    "noua zeelanda", "africa de sud", "haiti", "curacao", "capul verde", "irak", "panama"
]

# 2. Încărcăm datele
try:
    df = pd.read_excel("RATING_FINAL.xlsx")
except FileNotFoundError:
    print("❌ Eroare: Nu găsesc fișierul RATING_FINAL.xlsx.")

# 3. Calculăm scorurile pentru JUCĂTORI (Top 25)
scoruri_jucatori = []

# Extragem din tabel doar coloanele care NU sunt țări, URL sau Text_Stire
coloane_jucatori = [col for col in df.columns if col.lower() not in lista_tari and col not in ['URL', 'Text_Stire']]

for jucator in coloane_jucatori:
    scor_total = df[jucator].sum()
    if scor_total > 0:
        scoruri_jucatori.append({'Jucator': str(jucator).title(), 'Scor_Rating': scor_total})

# Transformăm în tabel, sortăm și reținem strict TOP 25
df_viz = pd.DataFrame(scoruri_jucatori).sort_values('Scor_Rating', ascending=False).head(25).reset_index(drop=True)

# TRUCUL DRAMATIC: Ridicăm scorul la pătrat doar pentru afișarea bulelor
df_viz['Scor_Dramatic'] = df_viz['Scor_Rating'] ** 2

# 4. Algoritm pentru evitarea suprapunerii (Pack Circles)
def pack_circles(data):
    # Folosim scorul dramatic pentru calculul razei, distanțându-le mai bine
    radii = np.sqrt(data['Scor_Dramatic'].values)
    x = np.zeros(len(radii))
    y = np.zeros(len(radii))

    for i in range(1, len(radii)):
        angle = i * 137.508  # Golden angle
        r_pos = radii[0] * 0.5
        collision = True
        while collision:
            r_pos += 0.2
            curr_x = r_pos * np.cos(np.radians(angle))
            curr_y = r_pos * np.sin(np.radians(angle))

            collision = False
            for j in range(i):
                dist = np.sqrt((curr_x - x[j])**2 + (curr_y - y[j])**2)
                if dist < (radii[i] + radii[j]) * 1.05:
                    collision = True
                    break
            x[i], y[i] = curr_x, curr_y
    return x, y

df_viz['X'], df_viz['Y'] = pack_circles(df_viz)
df_viz['Scor_Afisat'] = df_viz['Scor_Rating'].round(2)

# 5. Generarea graficului
fig = px.scatter(
    df_viz, x="X", y="Y",
    size="Scor_Dramatic", # Mărimea bulei se bazează pe scara dramatică
    color="Jucator",
    text="Jucator",
    hover_name="Jucator",
    # Ascundem "Scor_Dramatic" din popup ca să nu deruteze și afișăm doar datele reale
    hover_data={"Scor_Afisat": True, "X": False, "Y": False, "Scor_Dramatic": False, "Scor_Rating": False, "Jucator": False},
    size_max=85, # Mărit pentru un contrast vizual mai mare
    template="plotly_dark",
    title="🌟 TOP 25 Jucători Global 🌟"
)

fig.update_xaxes(visible=False, range=[df_viz['X'].min()-10, df_viz['X'].max()+10])
fig.update_yaxes(visible=False, range=[df_viz['Y'].min()-10, df_viz['Y'].max()+10])
fig.update_traces(textposition='middle center', textfont_size=11, textfont_color='white')
fig.update_layout(
    showlegend=False,
    margin=dict(t=50, b=0, l=0, r=0),
    hoverlabel=dict(bgcolor="white", font_size=14)
)

fig.show()

###########################################################
# @title Celula: Clasament Descrescător Top 25 Jucători
import plotly.express as px

# Pregătim datele pentru graficul de bare
df_bar_viz = df_viz.copy()
df_bar_viz['Scor'] = df_bar_viz['Scor_Rating'].round(2)

# Generăm graficul
fig_bar = px.bar(
    df_bar_viz,
    x="Jucator",
    y="Scor",
    color="Scor",
    text="Scor",
    color_continuous_scale="Viridis",
    template="plotly_dark",
    title="📊 Clasamentul Top 25 Jucători 📊",
    labels={"Scor": "Scor", "Jucator": "Jucător"}
)

fig_bar.update_traces(texttemplate='%{text}', textposition='outside')
fig_bar.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
    coloraxis_showscale=False,
    margin=dict(t=50, b=100, l=50, r=50)
)

fig_bar.show()

In [17]:
# @title Celula: Harta Interactivă + Grafic Scoruri (Dark Mode)
!pip install geopandas wordcloud ipywidgets --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from wordcloud import WordCloud
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io
import warnings

warnings.filterwarnings('ignore')

# 1. Încărcăm baza de date mondială (link direct)
url_harta = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
harta_lumii = gpd.read_file(url_harta)
harta_lumii.columns = harta_lumii.columns.str.lower()

# 2. Dicționar de traducere
traducere_harti = {
    "franta": "France", "argentina": "Argentina", "brazilia": "Brazil",
    "portugalia": "Portugal", "anglia": "United Kingdom", "spania": "Spain",
    "germania": "Germany", "olanda": "Netherlands", "croatia": "Croatia",
    "belgia": "Belgium", "uruguay": "Uruguay", "maroc": "Morocco",
    "senegal": "Senegal", "columbia": "Colombia", "norvegia": "Norway",
    "suedia": "Sweden", "turcia": "Turkey", "egipt": "Egypt",
    "sua": "United States of America", "elvetia": "Switzerland",
    "coreea de sud": "South Korea", "mexic": "Mexico", "japonia": "Japan",
    "canada": "Canada", "ghana": "Ghana", "ecuador": "Ecuador"
}

# 3. Extragem datele din tabel
try:
    df = pd.read_excel("RATING_FINAL.xlsx")
    df_scoruri = df.drop(columns=['URL', 'Text_Stire']).sum().to_dict()
except Exception as e:
    print("❌ Eroare la citirea Excel-ului.")
    df_scoruri = {}

mapare_jucatori = {}
for jucator, tara in mapare_jucator_tara.items():
    if tara not in mapare_jucatori:
        mapare_jucatori[tara] = []
    mapare_jucatori[tara].append(jucator)

# 4. Funcția care desenează Harta + Graficul
def deseneaza_harta_jucatori(tara_selectata):
    if tara_selectata not in mapare_jucatori:
        print(f"Nu am jucători pentru {tara_selectata}.")
        return

    nume_harta_en = traducere_harti.get(tara_selectata, tara_selectata.capitalize())
    forma_tara = harta_lumii[harta_lumii.name == nume_harta_en].copy()

    if forma_tara.empty:
        print(f"❌ Nu am găsit forma geografică pentru {tara_selectata}.")
        return

    geom = forma_tara.geometry.iloc[0]
    if geom.geom_type == 'MultiPolygon':
        poligon_principal = max(geom.geoms, key=lambda a: a.area)
        forma_tara = gpd.GeoDataFrame(geometry=[poligon_principal], crs=forma_tara.crs)

    # Creăm Masca pentru WordCloud (Aici rămâne alb-negru pt că așa cere masca)
    fig_mask, ax_mask = plt.subplots(figsize=(8, 8))
    forma_tara.plot(ax=ax_mask, color='black')
    ax_mask.axis('off')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', facecolor='white', bbox_inches='tight')
    plt.close(fig_mask)
    buf.seek(0)
    masca_pixeli = np.array(Image.open(buf))

    frecvente = {}
    scoruri_reale = {}
    for jucator in mapare_jucatori[tara_selectata]:
        scor = df_scoruri.get(jucator, 0)
        scoruri_reale[jucator.title()] = scor
        frecvente[jucator.title()] = max(0.1, scor)

    # Generăm WordCloud-ul cu setări Dark Mode
    wc = WordCloud(
        background_color='#121212', # Fundal negru/gri foarte închis
        mask=masca_pixeli,
        contour_width=3,
        contour_color='#4facfe',    # Un albastru luminos pentru contur pe negru
        colormap='Set3',            # Paletă pastelată luminoasă pentru text
        max_words=200
    )
    wc.generate_from_frequencies(frecvente)

    jucatori_sortati = sorted(scoruri_reale.items(), key=lambda item: item[1], reverse=False)
    nume_jucatori = [x[0] for x in jucatori_sortati]
    valori_scoruri = [x[1] for x in jucatori_sortati]

    # --- DESENĂM ECRANUL ÎMPĂRȚIT ÎN 2 (DARK MODE) ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), gridspec_kw={'width_ratios': [1, 1.2]})

    # Setăm fundalul figurii principale la negru
    culoare_fundal = '#121212'
    fig.patch.set_facecolor(culoare_fundal)

    # PARTEA STÂNGĂ: Harta WordCloud
    ax1.imshow(wc, interpolation='bilinear')
    ax1.set_title(f"Harta: {tara_selectata.upper()}", fontsize=22, pad=20, fontweight='bold', color='white')
    ax1.axis('off')

    # PARTEA DREAPTĂ: Graficul cu Bare
    ax2.set_facecolor(culoare_fundal) # Fundalul axelor

    culori = ['#2ca02c' if v > 0 else ('#d62728' if v < 0 else '#7f7f7f') for v in valori_scoruri]
    bare = ax2.barh(nume_jucatori, valori_scoruri, color=culori, edgecolor='white', alpha=0.9)

    # Stilizăm textul din dreapta în alb
    ax2.set_title("Clasament Sentiment (Model RoBERTa)", fontsize=22, pad=20, fontweight='bold', color='white')
    ax2.set_xlabel("Scor AI Total", fontsize=14, color='white')
    ax2.tick_params(axis='both', labelsize=12, colors='white')
    ax2.grid(axis='x', linestyle='--', alpha=0.3, color='lightgray') # Grid mai subtil

    # Stilizăm marginile graficului în alb
    for spine in ax2.spines.values():
        spine.set_color('white')

    # Adăugăm scorurile scrise pe fiecare bară cu text alb
    ax2.bar_label(bare, fmt='%.2f', padding=5, fontsize=12, fontweight='bold', color='white')

    plt.tight_layout()
    plt.show()

# 5. Interfața cu Dropdown
tari_disponibile = sorted(list(traducere_harti.keys()))
dropdown_tari = widgets.Dropdown(
    options=tari_disponibile,
    description='Selectează Țara:',
    style={'description_width': 'initial'}
)
output_grafic = widgets.Output()

def la_schimbare_dropdown(change):
    with output_grafic:
        clear_output(wait=True)
        deseneaza_harta_jucatori(change.new)

dropdown_tari.observe(la_schimbare_dropdown, names='value')

display(dropdown_tari, output_grafic)
with output_grafic:
    deseneaza_harta_jucatori(tari_disponibile[0])

Dropdown(description='Selectează Țara:', options=('anglia', 'argentina', 'belgia', 'brazilia', 'canada', 'colu…

Output()

In [23]:
# @title Celula: Proporția Sentimentelor pe Echipe (100% Stacked Bar)
!pip install plotly pandas --quiet

import pandas as pd
import plotly.express as px

# 1. Lista țărilor de analizat
lista_tari = [
    "franta", "argentina", "brazilia", "portugalia", "anglia", "spania", "germania",
    "olanda", "croatia", "belgia", "uruguay", "maroc", "senegal", "columbia",
    "norvegia", "suedia", "turcia", "egipt", "sua", "elvetia", "coreea de sud",
    "mexic", "japonia", "canada", "ghana", "ecuador", "coasta de fildes", "algeria",
    "austria", "cehia", "scotia", "iran", "arabia saudita", "australia", "tunisia",
    "paraguay", "qatar", "bosnia si hertegovina", "iordania", "rd congo", "uzbekistan",
    "noua zeelanda", "africa de sud", "haiti", "curacao", "capul verde", "irak", "panama"
]

# 2. Încărcăm datele
try:
    df = pd.read_excel("RATING_FINAL.xlsx")
except FileNotFoundError:
    print("❌ Eroare: Nu găsesc fișierul RATING_FINAL.xlsx.")

date_proporii = []

# 3. Procesarea matematică: Separăm Pozitivul de Negativ
for tara in lista_tari:
    if tara in df.columns:
        scoruri = df[tara]

        # Adunăm separat scorurile > 0 și cele < 0 (folosim valoarea absolută pentru calcule)
        scor_pozitiv = scoruri[scoruri > 0].sum()
        scor_negativ = abs(scoruri[scoruri < 0].sum())

        total_absolut = scor_pozitiv + scor_negativ

        # Analizăm doar echipele care au fost menționate (evităm împărțirea la 0)
        if total_absolut > 0:
            pct_pozitiv = (scor_pozitiv / total_absolut) * 100
            pct_negativ = (scor_negativ / total_absolut) * 100

            date_proporii.append({
                'Tara': tara.capitalize(),
                'Pozitiv (%)': pct_pozitiv,
                'Negativ (%)': pct_negativ
            })

# Transformăm în DataFrame și sortăm după sentimentul pozitiv
df_prop = pd.DataFrame(date_proporii)
df_prop = df_prop.sort_values(by='Pozitiv (%)', ascending=True)

# 4. Pregătim datele pentru structura Plotly (Melt format)
df_viz = df_prop.melt(
    id_vars=['Tara'],
    value_vars=['Pozitiv (%)', 'Negativ (%)'],
    var_name='Tip Sentiment',
    value_name='Procentaj'
)

# Rotunjim procentajele pentru a le afișa frumos pe grafic
df_viz['Procentaj_Afisat'] = df_viz['Procentaj'].apply(lambda x: f"{x:.1f}%")

# 5. Generarea Graficului 100% Stacked Bar
fig = px.bar(
    df_viz,
    y="Tara",
    x="Procentaj",
    color="Tip Sentiment",
    orientation='h',
    color_discrete_map={'Pozitiv (%)': '#2ca02c', 'Negativ (%)': '#d62728'}, # Verde pt Pozitiv, Roșu pt Negativ
    text="Procentaj_Afisat",
    template="plotly_dark",
    title="⚖️ Polarizarea Sentimentelor pe Echipe (Pozitiv vs. Negativ) ⚖️"
)

# Ajustări vizuale
fig.update_layout(
    barmode='stack',       # Forțează barele să stea una în prelungirea celeilalte
    xaxis_title="Proporție (%)",
    yaxis_title="",
    height=max(600, len(df_prop) * 25), # Înălțimea se ajustează automat în funcție de câte echipe ai
    margin=dict(l=100, r=40, t=70, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Setăm textul să fie centrat în interiorul barelor
fig.update_traces(textposition='inside', insidetextanchor='middle', textfont_size=12, textfont_color='white')

# Afișăm graficul
fig.show()